# LoomGuard: carpet model (PatchCore export)

STFPM localises carpet defects well (pixel AUROC 0.98) but its image-level score is unreliable on carpet (image AUROC 0.77). PatchCore reached 0.99 image AUROC on carpet, so the app serves carpet with PatchCore instead: per-category model routing.

This notebook fits PatchCore on carpet only, exports it to ONNX (the memory bank is a separate `bank.npy` input, keeping the model file under GitHub's 100 MB limit), and **calibrates and evaluates through the exported ONNX model itself**, so the reported numbers are exactly what the app produces.

Same setup as before: GPU on, Internet on, MVTec AD added as input. Then Run All (about 10 minutes).

In [ ]:
%pip install -q onnx onnxruntime onnxscript

In [ ]:
import os, glob, json, time, random, math, shutil, platform
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models
from sklearn.metrics import roc_auc_score, roc_curve, precision_recall_fscore_support, precision_recall_curve
import matplotlib
import matplotlib.pyplot as plt

SMOKE = os.environ.get("LOOMGUARD_SMOKE") == "1"   # local CI only: synthetic data, no downloads
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "")

In [ ]:
CATEGORIES      = ["carpet", "grid", "leather", "tile", "wood"]  # texture categories of MVTec AD
DEPLOY_CATEGORY = "carpet"     # the fabric model the demo app opens with
IMG_SIZE        = 256
SEED            = 42
VAL_FRACTION    = 0.10         # defect-free images held out for threshold calibration
STFPM_EPOCHS    = 100
STFPM_BATCH     = 32
STFPM_LR        = 0.4          # SGD, as in the STFPM paper
PATCHCORE_RATIO = 0.10         # coreset keeps 10% of patch features
THRESH_SIGMAS   = 3.0          # threshold = mean + 3*std of calibration (normal) scores
TOPK_FRACTION   = 0.01         # image score = mean of the hottest 1% of pixels (robust to single noisy pixels)
STFPM_RESTARTS  = 2            # train twice per category, keep the run with lower held-out loss (label-free)
CALIB_ON_DEPLOYMENT_NORMALS = True  # calibrate on half of the test-time good images (see section 7)
OUT             = "/kaggle/working/loomguard"

if SMOKE:
    CATEGORIES, DEPLOY_CATEGORY = ["carpet"], "carpet"
    IMG_SIZE, STFPM_EPOCHS, STFPM_BATCH = 64, 2, 4
    OUT = os.environ.get("LOOMGUARD_OUT", "./smoke_out")

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
seed_everything(SEED)

for sub in ["models", "figures", "samples"]:
    os.makedirs(os.path.join(OUT, sub), exist_ok=True)

CATEGORIES = ["carpet"]
PATCHCORE_RATIO = 0.01   # 1% coreset keeps bank.npy small (~16 MB) for the app
OUT = os.environ.get("LOOMGUARD_OUT", "./smoke_out_pc") if SMOKE else "/kaggle/working/loomguard_carpet"
os.makedirs(os.path.join(OUT, "models", "carpet"), exist_ok=True)

In [ ]:
def find_mvtec_root():
    if os.environ.get("LOOMGUARD_DATA"):
        return os.environ["LOOMGUARD_DATA"]
    for dirpath, dirnames, _ in os.walk("/kaggle/input"):
        if "carpet" in dirnames and os.path.isdir(os.path.join(dirpath, "carpet", "train", "good")):
            return dirpath
        if dirpath.count(os.sep) > 7:
            dirnames[:] = []
    raise FileNotFoundError("MVTec AD not found. Use 'Add Input' and attach an MVTec AD dataset.")

DATA_ROOT = find_mvtec_root()
CATEGORIES = [c for c in CATEGORIES if os.path.isdir(os.path.join(DATA_ROOT, c, "train", "good"))]
print("Data root:", DATA_ROOT)
print("Categories found:", CATEGORIES)

In [ ]:
IMG_EXT = (".png", ".jpg", ".jpeg", ".bmp")

def list_images(folder):
    return sorted(p for p in glob.glob(os.path.join(folder, "*")) if p.lower().endswith(IMG_EXT))

def build_split(cat):
    root = os.path.join(DATA_ROOT, cat)
    good = list_images(os.path.join(root, "train", "good"))
    rng = random.Random(SEED); rng.shuffle(good)
    n_val = max(5, int(len(good) * VAL_FRACTION))
    val, train = good[:n_val], good[n_val:]
    test = []
    for defect in sorted(os.listdir(os.path.join(root, "test"))):
        for p in list_images(os.path.join(root, "test", defect)):
            stem = os.path.splitext(os.path.basename(p))[0]
            mask = os.path.join(root, "ground_truth", defect, stem + "_mask.png")
            test.append({"path": p, "defect": defect, "label": int(defect != "good"),
                         "mask": mask if defect != "good" and os.path.exists(mask) else None})
    return train, val, test

def load_rgb(path, size=IMG_SIZE):
    img = Image.open(path).convert("RGB").resize((size, size), Image.BILINEAR)
    return torch.from_numpy(np.asarray(img, dtype=np.float32) / 255.0).permute(2, 0, 1)

def load_mask(path, size=IMG_SIZE):
    if path is None:
        return np.zeros((size, size), dtype=np.uint8)
    m = Image.open(path).convert("L").resize((size, size), Image.NEAREST)
    return (np.asarray(m) > 127).astype(np.uint8)

def load_stack(paths):
    return torch.stack([load_rgb(p) for p in paths])

tr, va, te = build_split(CATEGORIES[0])
print(f"{CATEGORIES[0]}: train={len(tr)} val={len(va)} test={len(te)} "
      f"(defective={sum(t['label'] for t in te)}, types={sorted({t['defect'] for t in te})})")

In [ ]:
def topk_score(amap):
    # Mean of the hottest TOPK_FRACTION of pixels. A single max pixel is too noisy on fuzzy textures like carpet.
    k = max(1, int(TOPK_FRACTION * amap.shape[-1] * amap.shape[-2]))
    return amap.flatten(1).topk(k, dim=1).values.mean(1)

class Normalize(nn.Module):
    def __init__(self):
        super().__init__()
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std",  torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))
    def forward(self, x):
        return (x - self.mean) / self.std

class GaussianBlur(nn.Module):
    # Separable Gaussian blur written with conv2d so it exports cleanly to ONNX.
    def __init__(self, sigma=4.0):
        super().__init__()
        radius = int(4 * sigma + 0.5)
        x = torch.arange(-radius, radius + 1, dtype=torch.float32)
        k = torch.exp(-x ** 2 / (2 * sigma ** 2)); k = k / k.sum()
        self.radius = radius
        self.register_buffer("kh", k.view(1, 1, 1, -1))
        self.register_buffer("kv", k.view(1, 1, -1, 1))
    def forward(self, x):
        r = self.radius
        x = F.conv2d(F.pad(x, (r, r, 0, 0), mode="replicate"), self.kh)
        return F.conv2d(F.pad(x, (0, 0, r, r), mode="replicate"), self.kv)

def imagenet_weights(enum):
    return None if SMOKE else enum.IMAGENET1K_V1

class ResNetPyramid(nn.Module):
    # Returns layer1, layer2, layer3 feature maps of a torchvision ResNet.
    def __init__(self, net):
        super().__init__()
        self.stem = nn.Sequential(net.conv1, net.bn1, net.relu, net.maxpool)
        self.layer1, self.layer2, self.layer3 = net.layer1, net.layer2, net.layer3
    def forward(self, x):
        f1 = self.layer1(self.stem(x)); f2 = self.layer2(f1); f3 = self.layer3(f2)
        return [f1, f2, f3]

class STFPM(nn.Module):
    # Student-Teacher Feature Pyramid Matching (Wang et al., 2021).
    def __init__(self, img_size=IMG_SIZE):
        super().__init__()
        self.img_size = img_size
        self.norm = Normalize()
        self.teacher = ResNetPyramid(models.resnet18(weights=imagenet_weights(models.ResNet18_Weights)))
        self.student = ResNetPyramid(models.resnet18(weights=None))
        for p in self.teacher.parameters():
            p.requires_grad_(False)
        self.blur = GaussianBlur(4.0)

    def train(self, mode=True):
        super().train(mode); self.teacher.eval()   # teacher BN statistics stay frozen
        return self

    def pyramids(self, x):
        x = self.norm(x)
        with torch.no_grad():
            t = self.teacher(x)
        return t, self.student(x)

    @staticmethod
    def loss(t_feats, s_feats):
        return sum(0.5 * (F.normalize(t, dim=1) - F.normalize(s, dim=1)).pow(2).sum(1).mean()
                   for t, s in zip(t_feats, s_feats))

    def forward(self, x):
        # Returns (anomaly map [B,1,H,W], image score [B])
        t_feats, s_feats = self.pyramids(x)
        amap = None
        for t, s in zip(t_feats, s_feats):
            m = 0.5 * (F.normalize(t, dim=1) - F.normalize(s, dim=1)).pow(2).sum(1, keepdim=True)
            m = F.interpolate(m, size=(self.img_size, self.img_size), mode="bilinear", align_corners=False)
            amap = m if amap is None else amap * m
        amap = self.blur(amap)
        return amap, topk_score(amap)

In [ ]:
class PatchCore(nn.Module):
    # PatchCore (Roth et al., 2022): locally aware patch features + greedy coreset memory bank.
    def __init__(self, img_size=IMG_SIZE, coreset_ratio=PATCHCORE_RATIO):
        super().__init__()
        net = (models.resnet18(weights=None) if SMOKE else
               models.wide_resnet50_2(weights=models.Wide_ResNet50_2_Weights.IMAGENET1K_V1))
        self.backbone = ResNetPyramid(net).eval()
        for p in self.backbone.parameters():
            p.requires_grad_(False)
        self.norm, self.blur = Normalize(), GaussianBlur(4.0)
        self.img_size, self.ratio = img_size, coreset_ratio
        self.register_buffer("bank", torch.empty(0))

    @torch.no_grad()
    def embed(self, x):
        _, f2, f3 = self.backbone(self.norm(x))
        f2 = F.avg_pool2d(f2, 3, 1, 1)
        f3 = F.interpolate(F.avg_pool2d(f3, 3, 1, 1), size=f2.shape[-2:], mode="bilinear", align_corners=False)
        return torch.cat([f2, f3], 1)                       # [B, C, h, w]

    @torch.no_grad()
    def fit(self, images, batch=16):
        feats = []
        for i in range(0, len(images), batch):
            e = self.embed(images[i:i + batch].to(DEVICE))
            feats.append(e.permute(0, 2, 3, 1).reshape(-1, e.shape[1]))
        feats = torch.cat(feats)
        idx = greedy_coreset(feats, self.ratio)
        self.bank = feats[idx].contiguous()
        return feats.shape[0], self.bank.shape[0]

    @torch.no_grad()
    def forward(self, x):
        e = self.embed(x)
        b, c, h, w = e.shape
        q = e.permute(0, 2, 3, 1).reshape(-1, c)
        d = torch.cat([torch.cdist(q[i:i + 4096], self.bank).amin(1) for i in range(0, len(q), 4096)])
        m = d.view(b, 1, h, w)
        m = self.blur(F.interpolate(m, size=(self.img_size, self.img_size), mode="bilinear", align_corners=False))
        return m, topk_score(m)

@torch.no_grad()
def greedy_coreset(x, ratio, proj_dim=128):
    # Greedy k-center selection on a random projection (keeps the GPU busy, no host syncs in the loop).
    n = x.shape[0]; k = max(1, int(n * ratio))
    g = torch.Generator(device=x.device).manual_seed(SEED)
    z = x @ (torch.randn(x.shape[1], proj_dim, device=x.device, generator=g) / math.sqrt(proj_dim))
    sel = torch.empty(k, dtype=torch.long, device=x.device)
    sel[0] = 0
    min_d = (z - z[0]).pow(2).sum(1)
    for j in range(1, k):
        i = torch.argmax(min_d)
        sel[j] = i
        min_d = torch.minimum(min_d, (z - z[i]).pow(2).sum(1))
    return sel

In [ ]:
class PatchCoreDeploy(nn.Module):
    # Exportable PatchCore: nearest-neighbour distance written with matmuls; the memory bank is an input.
    def __init__(self, pc):
        super().__init__()
        self.pc = pc
    def forward(self, image, bank):
        e = self.pc.embed(image)
        c, h, w = e.shape[1], e.shape[2], e.shape[3]
        q = e.permute(0, 2, 3, 1).reshape(-1, c)
        d2 = (q * q).sum(1, keepdim=True) - 2 * (q @ bank.t()) + (bank * bank).sum(1).unsqueeze(0)
        d = d2.clamp_min(0).amin(1).sqrt()
        m = d.reshape(-1, 1, h, w)
        m = self.pc.blur(F.interpolate(m, size=(self.pc.img_size, self.pc.img_size), mode="bilinear", align_corners=False))
        return m, topk_score(m)

In [ ]:
import onnxruntime as ort
seed_everything(SEED)
cat = "carpet"
train_p, val_p, test = build_split(cat)
good_idx = [i for i, t in enumerate(test) if t["label"] == 0]
random.Random(SEED).shuffle(good_idx)
calib_idx = set(good_idx[:len(good_idx) // 2])          # identical split to the main notebook
calib_p = [test[i]["path"] for i in sorted(calib_idx)]
test = [t for i, t in enumerate(test) if i not in calib_idx]
print(f"train={len(train_p)} calibration={len(calib_p)} test={len(test)}")

t0 = time.time()
pc = PatchCore().to(DEVICE)
n_patches, n_bank = pc.fit(load_stack(train_p))
print(f"memory bank: {n_bank:,} of {n_patches:,} patches ({time.time() - t0:.0f}s)")

mdir = os.path.join(OUT, "models", cat)
bank = pc.bank.float().cpu().numpy()
np.save(os.path.join(mdir, "bank.npy"), bank)
deploy = PatchCoreDeploy(pc).eval().cpu()
onnx_path = os.path.join(mdir, "patchcore.onnx")
dummy = (torch.rand(1, 3, IMG_SIZE, IMG_SIZE), torch.from_numpy(bank))
kw = dict(input_names=["image", "bank"], output_names=["anomaly_map", "score"], opset_version=17)
try:
    torch.onnx.export(deploy, dummy, onnx_path, dynamo=False, **kw)
except TypeError:
    torch.onnx.export(deploy, dummy, onnx_path, **kw)
size_mb = os.path.getsize(onnx_path) / 2**20
print(f"ONNX: {size_mb:.1f} MB")

In [ ]:
sess = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
def run(path):
    x = load_rgb(path).numpy()[None]
    m, s = sess.run(None, {"image": x, "bank": bank})
    return m[0, 0], float(s[0])

with torch.no_grad():
    ref = deploy(load_rgb(test[0]["path"])[None], torch.from_numpy(bank))[1].item()
parity = abs(run(test[0]["path"])[1] - ref) / (abs(ref) + 1e-12)
print(f"ONNX vs PyTorch relative error: {parity:.2e}")

cal = [run(p) for p in calib_p]
cal_maps = np.stack([m for m, _ in cal]); cal_scores = np.array([s for _, s in cal])
mu, sd = float(cal_scores.mean()), float(cal_scores.std() + 1e-8)
calib = {"threshold": mu + THRESH_SIGMAS * sd, "val_mean": mu, "val_std": sd,
         "pixel_p50": float(np.median(cal_maps)), "pixel_max": float(np.percentile(cal_maps, 99.9))}

times, maps, scores = [], [], []
for t in test:
    t1 = time.perf_counter(); m, s = run(t["path"]); times.append(time.perf_counter() - t1)
    maps.append(m); scores.append(s)
maps, scores = np.stack(maps), np.array(scores)
masks = np.stack([load_mask(t["mask"]) for t in test])
labels = np.array([t["label"] for t in test])
pred = scores > calib["threshold"]
p, r, f1, _ = precision_recall_fscore_support(labels, pred, average="binary", zero_division=0)
per_type = {d: float(pred[[i for i, t in enumerate(test) if t["defect"] == d]].mean()) if d != "good"
            else float(1 - pred[[i for i, t in enumerate(test) if t["defect"] == d]].mean())
            for d in sorted({t["defect"] for t in test})}
metrics = {"image_auroc": float(roc_auc_score(labels, scores)),
           "pixel_auroc": float(roc_auc_score(masks.reshape(-1), maps.reshape(-1))),
           "precision": float(p), "recall": float(r), "f1": float(f1),
           "false_alarm_rate": float(pred[labels == 0].mean()), "per_defect_detection": per_type,
           "cpu_latency_ms": float(np.median(times) * 1000), "onnx_size_mb": round(size_mb, 1),
           "bank_size": int(n_bank), "onnx_rel_error": float(parity)}
meta = {"category": cat, "model": "patchcore-wide_resnet50_2", "onnx_file": "patchcore.onnx", "bank_file": "bank.npy",
        "img_size": IMG_SIZE, "input": "RGB float32 in [0,1], NCHW", **calib, "metrics": metrics,
        "trained_on": "MVTec AD (CC BY-NC-SA 4.0)"}
with open(os.path.join(mdir, "meta.json"), "w") as f:
    json.dump(meta, f, indent=2)

print(json.dumps({k: (round(v, 4) if isinstance(v, float) else v) for k, v in metrics.items()}, indent=1))
ok = metrics["image_auroc"] > 0.95 and metrics["f1"] > 0.8 and metrics["false_alarm_rate"] <= 0.15
print("\nHEALTH CHECK:", "carpet PatchCore is good." if ok else "carpet result is weak, send me the numbers above.")
archive = shutil.make_archive(OUT + "_artifacts", "zip", OUT)
print("Download:", archive)